# 05 — Redes convolucionales (CNN)

Estudiaremos una CNN pequeña entrenada desde cero. Las convoluciones comparten pesos entre posiciones y conservan las relaciones espaciales de los píxeles. Compararemos esta representación con la Dense de 04.

La comparación principal usará RGB64 y un número de parámetros dentro de ±10 % del baseline Dense. BatchNorm, dropout, pooling y bloques residuales son posibles ampliaciones que se estudiarán por separado, según el coste medido.

> **Plan pendiente de implementación.** Este cuaderno contiene explicaciones y propuestas. No contiene código ejecutado ni resultados de entrenamiento.

## Vía A · CNN desde cero y posible atención

CNN es una arquitectura, mientras que scratch describe cómo inicializamos sus pesos. Aquí se estudia la convolución espacial en imágenes; el notebook 07 reutilizará pesos preentrenados como otra vía. Que la CNN pequeña gane o pierda frente a Dense importa menos que explicar qué aporta su estructura local y el peso compartido.

La pista «CNN + attention layer» puede interpretarse como una extensión: añadir atención sobre canales o posiciones, o transformar posiciones de un mapa de características en tokens. No es un requisito para que una red sea CNN. Una CNN1D sobre word embeddings pertenece a la rama NLP opcional de 11 y no es este experimento visual. Una híbrida CNN + LSTM/GRU también es posible, pero no convierte las unidades recurrentes en convoluciones. Si se añade atención, comparar con la misma CNN sin ella mediante una ablación, sin prometer mejoras. [Fundamento del mecanismo de atención](https://arxiv.org/abs/1706.03762).

## Protocolo común del proyecto

Primero definimos las prendas que queremos clasificar. Después aplicamos los mismos controles de calidad a todas las fotos y conservamos **todas las que los superan**, aunque unas clases tengan más ejemplos. No se fija un número total ni se igualan las clases con cupos.

Las particiones se construyen sobre todos los grupos de producto elegibles. Los productos ya utilizados durante el desarrollo conservan su exposición: un producto de train no se convierte en un supuesto test nuevo. Los grupos nuevos completan aproximadamente 70/15/15 por clase. Se publican los recuentos reales.

**Antes de recortar train por coste, hay que medirlo.** El baseline usa todos los productos elegibles; 04 registra una época piloto y la estimación de tiempo/memoria. Si fuese necesario reducir datos en otro experimento, se justificaría con esa medición, solo en train y conservando validation/test. Una fotografía fuera de una muestra de cálculo no sería una fotografía de mala calidad.

La referencia es `config/experimental_protocol_v1.json`: macro-F1 primaria, accuracy y resultados por clase; pérdida inicial sin pesos, sin aumentos; normalización solo con train a la resolución de entrada. Tres semillas y coste computacional. Los pesos de clase se estudian como ablación: con frecuencias desiguales ya no suponemos que sean casi uno. El test predictivo se reserva hasta congelar las decisiones. Las métricas describen las 26 clases elegidas tras los controles, no todo el catálogo.

**Filtro y reutilización.** En 02 se aplica el mismo criterio visual fijo a train, validation y test; las fotos conservadas no cambian de partición. El test se reserva para evaluación predictiva, aunque haya pasado controles e inspección visual. Se guardarán resultados y predicciones para recalcular métricas sin repetir ajustes. Cualquier extracción o entrenamiento costoso nuevo requerirá una activación explícita; este cuaderno todavía no implementa ese flujo.

## Objetivos

- Explicar convolución, activación, pooling y campo receptivo con ejemplos pequeños.
- Construir una CNN RGB64 con presupuesto comparable a la Dense, usando las mismas particiones.
- Comparar macro-F1, accuracy, errores por clase, parámetros, tiempo y memoria.
- Visualizar filtros y mapas de características sin atribuirles una interpretación causal automática.
- Priorizar pocas ampliaciones controladas; medir el coste antes de comprometer nuevos ajustes.

## Pasos previstos

1. Preparar RGB64 y documentar arquitectura y parámetros.
2. Medir una época piloto y registrar el presupuesto.
3. Entrenar la referencia y comparar con 04 bajo el protocolo común.
4. Estudiar las ampliaciones seleccionadas, cambiando un componente cada vez.
5. Interpretar curvas, errores y limitaciones. Una resolución diferente sería otra ablación, no el contraste principal.

## Entradas, salidas y material de clase

| | |
|---|---|
| **Entradas** | Datos de 02 y baseline Dense de 04 |
| **Salidas** | Métricas, predicciones, curvas y mapas de características en results/ |
| **Material de clase** | `PRDL_MLLB_CNN.pptx`, `PRDL_MLLB_RestNets_to_NeuralODE.pptx`, `MLLB_CNN_MNIST_in_Keras.ipynb`, `Gemini_MNIST_Pytorch_featuremaps.ipynb` |